# SETUP

In [ ]:
# Clone the NeMo Guardrails repository
!git clone https://github.com/NVIDIA-NeMo/Guardrails.git nemoguardrails
%cd nemoguardrails
!pip install -e .

fatal: destination path 'nemoguardrails' already exists and is not an empty directory.
/content/nemoguardrails/nemoguardrails/nemoguardrails/nemoguardrails
Obtaining file:///content/nemoguardrails/nemoguardrails/nemoguardrails/nemoguardrails
ERROR: file:///content/nemoguardrails/nemoguardrails/nemoguardrails/nemoguardrails does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
# Install NeMo Guardrails and its dependencies
!pip install nemoguardrails
!pip install langchain
!pip install langchain-nvidia-ai-endpoints
!pip install nest_asyncio

In [ ]:
# Validate install
import nest_asyncio
nest_asyncio.apply()

import os
import sys

print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
# Setup NVIDIA API KEY
from google.colab import userdata
import os

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

print("NVIDIA_API_KEY loaded:", bool(os.environ.get("NVIDIA_API_KEY")))
print("Key prefix:", os.environ["NVIDIA_API_KEY"][:6])

NVIDIA_API_KEY loaded: True
Key prefix: nvapi-


# DEFINE GUARDRAILS CONFIGURATION

In [ ]:
# Defining config

import os

!rm -rf config
!mkdir -p config

config_yml_content = """
models:
  - type: main
    engine: nvidia_ai_endpoints
    model: meta/llama-3.1-8b-instruct
    parameters:
      temperature: 0.0

instructions:
  - type: general
    content: |
      You are a driving license manual assistant.
      Only answer questions related to driving rules, road signs, parking, signaling, speed limits, traffic laws, and safe driving.
      If the user asks something unrelated, politely say you can only help with driving license manual questions.
      Do not encourage unsafe or illegal driving behavior.

sample_conversation: |
  user "hi"
    express greeting
  bot express greeting
    "Hello! How can I help you with the driving license manual today?"

rails:
  input:
    flows:
      - self check input
  output:
    flows:
      - self check output
"""

with open("config/config.yml", "w") as f:
    f.write(config_yml_content)

print(open("config/config.yml").read())


models:
  - type: main
    engine: nvidia_ai_endpoints
    model: meta/llama-3.1-8b-instruct
    parameters:
      temperature: 0.0

instructions:
  - type: general
    content: |
      You are a driving license manual assistant.
      Only answer questions related to driving rules, road signs, parking, signaling, speed limits, traffic laws, and safe driving.
      If the user asks something unrelated, politely say you can only help with driving license manual questions.
      Do not encourage unsafe or illegal driving behavior.

sample_conversation: |
  user "hi"
    express greeting
  bot express greeting
    "Hello! How can I help you with the driving license manual today?"

rails:
  input:
    flows:
      - self check input
  output:
    flows:
      - self check output



In [ ]:
# CREATE COLANG FILE
rails_co_content = """
define user express greeting
  "hello"
  "hi"
  "good morning"

define bot express greeting
  "Hello! How can I help you with the driving license manual today?"
  "Hi there! Ask me anything about driving rules."

define flow greeting
  user express greeting
  bot express greeting


define user express ask speed limit
  "What is the speed limit in residential areas?"
  "How fast can I drive in a neighborhood?"

define bot express answer speed limit
  "The speed limit in residential areas is typically 25 mph unless otherwise posted."

define flow get speed limit
  user express ask speed limit
  bot express answer speed limit


define user express ask signal turn
  "How to signal a turn?"
  "When should I use my turn signal?"

define bot express answer signal turn
  "Use your turn signal before turning or changing lanes. Many manuals recommend signaling at least 100 feet before the turn."

define flow get signal turn info
  user express ask signal turn
  bot express answer signal turn


define user express ask parallel parking
  "What are the rules for parallel parking?"
  "How do I parallel park?"

define bot express answer parallel parking
  "For parallel parking, stop beside the vehicle in front of the space, check traffic, reverse slowly, steer into the space, straighten the wheels, and adjust safely."

define flow get parallel parking info
  user express ask parallel parking
  bot express answer parallel parking


define user express ask road signs
  "Tell me about road signs."
  "What do road signs mean?"

define bot express answer road signs
  "Road signs use specific shapes, colors, and symbols to give warnings, rules, directions, and guidance."

define flow get road signs info
  user express ask road signs
  bot express answer road signs


define user express thanks
  "thank you"
  "thanks"

define bot express welcome
  "You're welcome!"

define flow thanks
  user express thanks
  bot express welcome
"""

with open("config/rails.co", "w") as f:
    f.write(rails_co_content)

print(open("config/rails.co").read())


define user express greeting
  "hello"
  "hi"
  "good morning"

define bot express greeting
  "Hello! How can I help you with the driving license manual today?"
  "Hi there! Ask me anything about driving rules."

define flow greeting
  user express greeting
  bot express greeting


define user express ask speed limit
  "What is the speed limit in residential areas?"
  "How fast can I drive in a neighborhood?"

define bot express answer speed limit
  "The speed limit in residential areas is typically 25 mph unless otherwise posted."

define flow get speed limit
  user express ask speed limit
  bot express answer speed limit


define user express ask signal turn
  "How to signal a turn?"
  "When should I use my turn signal?"

define bot express answer signal turn
  "Use your turn signal before turning or changing lanes. Many manuals recommend signaling at least 100 feet before the turn."

define flow get signal turn info
  user express ask signal turn
  bot express answer signal turn




In [ ]:
# Input/Output checks
prompts_yml_content = """
prompts:
  - task: self_check_input
    content: |
      Your task is to check if the user message is allowed.

      The bot is a driving license manual assistant.
      The user message is allowed only if it is about:
      - driving rules
      - traffic laws
      - road signs
      - safe driving
      - parking
      - speed limits
      - driver license manual topics

      The user message is not allowed if it asks about unrelated topics, jokes, cooking, movies, weather, politics, hacking, violence, or illegal behavior.

      User message: "{{ user_input }}"

      Should the user message be blocked?
      Answer only "yes" or "no".

  - task: self_check_output
    content: |
      Your task is to check if the bot response is safe.

      The bot response should not encourage:
      - unsafe driving
      - illegal behavior
      - drunk driving
      - ignoring traffic laws
      - cheating on a driving test

      Bot response: "{{ bot_response }}"

      Should the bot response be blocked?
      Answer only "yes" or "no".
"""

with open("config/prompts.yml", "w") as f:
    f.write(prompts_yml_content)

print(open("config/prompts.yml").read())


prompts:
  - task: self_check_input
    content: |
      Your task is to check if the user message is allowed.

      The bot is a driving license manual assistant.
      The user message is allowed only if it is about:
      - driving rules
      - traffic laws
      - road signs
      - safe driving
      - parking
      - speed limits
      - driver license manual topics

      The user message is not allowed if it asks about unrelated topics, jokes, cooking, movies, weather, politics, hacking, violence, or illegal behavior.

      User message: "{{ user_input }}"

      Should the user message be blocked?
      Answer only "yes" or "no".

  - task: self_check_output
    content: |
      Your task is to check if the bot response is safe.

      The bot response should not encourage:
      - unsafe driving
      - illegal behavior
      - drunk driving
      - ignoring traffic laws
      - cheating on a driving test

      Bot response: "{{ bot_response }}"

      Should the bot res

# LOAD AND TEST

In [ ]:
# Verify Guardrails loaded
from nemoguardrails import LLMRails, RailsConfig

config = RailsConfig.from_path("./config")
rails = LLMRails(config)

print("Guardrails loaded successfully.")

Guardrails loaded successfully.


In [ ]:
# User test 1
response = await rails.generate_async(messages=[
    {"role": "user", "content": "hi"}
])

print(response["content"])

I'm sorry, I can't respond to that.


In [ ]:
# User test 2
response = await rails.generate_async(messages=[
    {"role": "user", "content": "What is the speed limit in residential areas?"}
])

print(response["content"])

The speed limit in residential areas is typically 25 mph unless otherwise posted.


In [ ]:
# User test 3
response = await rails.generate_async(messages=[
    {"role": "user", "content": "Tell me a joke"}
])

print(response["content"])

I'm sorry, I can't respond to that.


In [ ]:
# User test 4
response = await rails.generate_async(messages=[
    {"role": "user", "content": "Is it okay to drink and drive if I am careful?"}
])

print(response["content"])

I'm sorry, I can't respond to that.
